#  Flight Delay Prediction
## Phase 3 — Exploratory Data Analysis (EDA)

---

| Field | Details |
|---|---|
| **Course** | COMP4381 – Data Science and Analytics, Spring 2026 |
| **Instructor** | Ahmed Sabbah |



---

## Table of Contents

1. [Introduction](#1-introduction)
2. [Dataset Source](#2-dataset-source)
3. [Import Libraries](#3-import-libraries)
4. [Load Dataset](#4-load-dataset)
5. [Dataset Structure](#5-dataset-structure)
6. [Dataset Dimensions](#6-dataset-dimensions)
7. [Column Names & Types](#7-column-names--types)
8. [Missing Values Analysis](#8-missing-values-analysis)
9. [Duplicate Records](#9-duplicate-records)
10. [Descriptive Statistics](#10-descriptive-statistics)
11. [Delay Analysis](#11-delay-analysis)
12. [Outlier Detection](#12-outlier-detection)
13. [Airport Analysis](#13-airport-analysis)
14. [Airline Distribution](#14-airline-distribution)


---
## 1. Introduction

This notebook presents a **comprehensive Exploratory Data Analysis (EDA)** of a large-scale flight delay dataset. The goal of this phase is to:

- Understand the structure and quality of the raw data
- Identify missing values, duplicates, and outliers
- Summarise key statistical properties of delay-related variables
- Prepare insights that will guide the feature engineering and modelling phases

>**Dataset size:** 1,747,627 flight records across 9 major US airports.

---
## 2. Dataset Source

| Property | Value |
|---|---|
| **Source** | Kaggle – Flight Delay Dataset |
| **Total Records** | 1,747,627 flights |
| **Total Features** | 16 columns |
| **Airports Covered** | ATL, DFW, JFK, LAX, ORD, BOS, MIA, SEA, SFO |
| **File Path** | `../data/raw/flight_delays.csv` |

The dataset contains scheduled and actual departure/arrival times, airline codes, origin and destination airports, and delay information in minutes.

---
## 3. Import Libraries

We use the following standard Python libraries:

- **`pandas`** — data loading, cleaning, and manipulation
- **`numpy`** — numerical operations and array handling
- **`matplotlib` / `seaborn`** — visualisation
- **`warnings`** — suppress non-critical runtime warnings

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', lambda x: f'{x:.2f}')

print('[OK] Libraries imported successfully!')

---
## 4. Load Dataset

The raw dataset is stored in `data/raw/flight_delays.csv` as per the project folder structure. We load it using `pandas.read_csv()` and confirm the shape immediately.

In [ ]:
df = pd.read_csv('../data/raw/flight_delays.csv')
print(f'[OK] Data loaded: {df.shape[0]:,} rows, {df.shape[1]:,} columns')

---
## 5. Dataset Structure

We preview the first few rows to understand how the data looks before any transformations. This helps spot obvious formatting issues or unexpected values early.

In [ ]:
print('=' * 70)
print('DATASET STRUCTURE — FIRST 5 ROWS')
print('=' * 70)
df.head()

---
## 6. Dataset Dimensions

We confirm the total number of rows and columns to verify the data loaded completely.

In [ ]:
print('=' * 70)
print('DATASET DIMENSIONS')
print('=' * 70)
print(f'  Rows    : {df.shape[0]:,}')
print(f'  Columns : {df.shape[1]:,}')

---
## 7. Column Names & Types

Understanding the data type of each column is essential before any cleaning step. Mismatched types (e.g. a numeric column stored as `object`) are a common source of errors.

In [ ]:
print('=' * 70)
print('COLUMN NAMES')
print('=' * 70)
for idx, col in enumerate(df.columns, 1):
    print(f'  {idx:2d}. {col}')

In [ ]:
print('=' * 70)
print('DATA TYPE DISTRIBUTION')
print('=' * 70)
for dtype, count in df.dtypes.value_counts().items():
    print(f'  {str(dtype):20s}: {count} column(s)')

In [ ]:
print('=' * 70)
print('COLUMN SUMMARY TABLE')
print('=' * 70)
desc = pd.DataFrame({
    'Column'  : df.columns,
    'Type'    : df.dtypes.values,
    'Missing' : df.isnull().sum().values
})
print(desc.to_string(index=False))

---
## 8. Missing Values Analysis

Missing data can introduce bias or cause model failures. Here we compute the **overall data completeness** as a percentage and flag any columns that require imputation or removal.

> A completeness score above **95%** is generally considered acceptable for tabular datasets.

In [ ]:
completeness = (1 - (df.isnull().sum().sum() / (len(df) * len(df.columns)))) * 100
print(f'  Data Completeness : {completeness:.2f}%')

missing_per_col = df.isnull().sum()
missing_cols = missing_per_col[missing_per_col > 0]
if missing_cols.empty:
    print('  [OK] No missing values detected in any column.')
else:
    print('\n  Columns with missing values:')
    print(missing_cols.to_string())

---
## 9. Duplicate Records

Duplicate rows can skew statistical summaries and model training. We check for fully duplicated rows and report the count.

In [ ]:
dups = df.duplicated().sum()
print(f'  Duplicate rows : {dups:,}')
if dups == 0:
    print('  [OK] No duplicate records found — data is unique.')
else:
    print(f'  [!] {dups:,} duplicate rows detected. Consider removing them before modelling.')

---
## 10. Descriptive Statistics

We compute standard summary statistics (count, mean, standard deviation, min, quartiles, max) for all **numeric columns**. This helps identify the range and spread of delay values and other continuous features.

In [ ]:
numeric_cols = df.select_dtypes(include=[np.number]).columns
df[numeric_cols].describe()

---
## 11. Delay Analysis

The primary variable of interest is `DelayMinutes`. Here we examine:

- **Mean delay** across all flights
- **Number of delayed flights** (where `DelayMinutes > 0`)
- **Proportion of delayed flights** relative to the total

This gives an initial picture of how frequently delays occur and how severe they tend to be.

In [ ]:
delay_stats = df['DelayMinutes'].describe()
delayed     = (df['DelayMinutes'] > 0).sum()
delay_pct   = (delayed / len(df)) * 100

print('=' * 70)
print('DELAY SUMMARY')
print('=' * 70)
print(f'  Mean Delay       : {delay_stats["mean"]:.2f} minutes')
print(f'  Median Delay     : {delay_stats["50%"]:.2f} minutes')
print(f'  Max Delay        : {delay_stats["max"]:.2f} minutes')
print(f'  Delayed Flights  : {delayed:,}  ({delay_pct:.2f}% of total)')

---
## 12. Outlier Detection

We use the **Interquartile Range (IQR) method** to detect outliers in `DelayMinutes`. A value is flagged as an outlier if it falls below `Q1 − 1.5×IQR` or above `Q3 + 1.5×IQR`.

Extreme delay values may represent genuine disruptions (e.g. severe weather, mechanical failure) or data entry errors — both require careful treatment in the modelling phase.

In [ ]:
Q1  = df['DelayMinutes'].quantile(0.25)
Q3  = df['DelayMinutes'].quantile(0.75)
IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

outliers = ((df['DelayMinutes'] < lower_bound) | (df['DelayMinutes'] > upper_bound)).sum()

print('=' * 70)
print('OUTLIER DETECTION — DelayMinutes (IQR Method)')
print('=' * 70)
print(f'  Q1 (25th pct)    : {Q1:.2f} min')
print(f'  Q3 (75th pct)    : {Q3:.2f} min')
print(f'  IQR              : {IQR:.2f} min')
print(f'  Lower Bound      : {lower_bound:.2f} min')
print(f'  Upper Bound      : {upper_bound:.2f} min')
print(f'  Outliers Found   : {outliers:,}')

---
## 13. Airport Analysis

We identify all unique airports appearing as either **origin** or **destination** in the dataset. Understanding which airports are included helps us assess geographic coverage and plan any merge with external airport metadata (e.g. OurAirports).

In [ ]:
origins      = df['Origin'].unique()
destinations = df['Destination'].unique()
all_airports = sorted(set(list(origins) + list(destinations)))

print('=' * 70)
print('AIRPORT COVERAGE')
print('=' * 70)
print(f'  Origin airports      : {len(origins)}')
print(f'  Destination airports : {len(destinations)}')
print(f'  Total unique airports: {len(all_airports)}')
print(f'  Airport codes        : {", ".join(all_airports)}')

---
## 14. Airline Distribution

We examine how flights are distributed across airlines. A heavily imbalanced distribution (one airline dominating) could introduce bias in the model, so this step informs any stratified sampling decisions later.

In [ ]:
airline_counts = df['Airline'].value_counts()

print('=' * 70)
print('AIRLINE DISTRIBUTION')
print('=' * 70)
for airline, count in airline_counts.items():
    pct = (count / len(df)) * 100
    bar = '█' * int(pct / 2)
    print(f'  {airline:6s} : {count:>9,}  ({pct:5.2f}%)  {bar}')